In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.svm import LinearSVC
import gc

In [3]:
print("--- 1. NẠP VÀ GỘP DỮ LIỆU TỪ CẢ 2 THÁNG (OCT & NOV) ---")
# Nạp dữ liệu từ cả 2 giai đoạn DWH sạch như cấu trúc dự án của bạn
df_oct = pd.read_csv('2019-Oct-Cleaned.csv')
df_nov = pd.read_csv('2019-Nov-Cleaned.csv')

# Gộp hai tập dữ liệu lại thành một để không bị thiếu thông tin học tập
df = pd.concat([df_oct, df_nov], ignore_index=True)

# Giải phóng bộ nhớ RAM của các tập dữ liệu đơn lẻ sau khi gộp
del df_oct, df_nov
gc.collect()

--- 1. NẠP VÀ GỘP DỮ LIỆU TỪ CẢ 2 THÁNG (OCT & NOV) ---


0

In [4]:
if 'event_time' in df.columns:
    df['event_time'] = pd.to_datetime(df['event_time'].str.replace(' UTC', ''), errors='coerce')
    df['hour'] = df['event_time'].dt.hour
    df['day_of_week'] = df['event_time'].dt.dayofweek

df['is_purchase'] = (df['event_type'] == 'purchase').astype(int)

# Mã hóa Label Encoding
categorical_cols = {'day_of_week': 'day_of_week_encoded', 'brand': 'brand_encoded', 'category_code': 'category_code_encoded'}
for col, encoded_col in categorical_cols.items():
    if col in df.columns:
        df[encoded_col] = df[col].astype(str).factorize()[0]

features = ['price', 'hour', 'day_of_week_encoded', 'brand_encoded', 'category_code_encoded']
if 'price' in df.columns:
    df['price'] = df['price'].fillna(df['price'].median())

X = df[features]
y = df['is_purchase']

In [5]:
print("--- 2. CHIA TẬP TRAIN/TEST VÀ CHUẨN HÓA DỮ LIỆU TỔNG ---")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- 3. CÂN BẰNG DỮ LIỆU BẰNG SMOTE CHO TOÀN BỘ TẬP TRAIN ---")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Giải phóng bộ nhớ RAM
del df, X, y, X_train, X_train_scaled
gc.collect()

--- 2. CHIA TẬP TRAIN/TEST VÀ CHUẨN HÓA DỮ LIỆU TỔNG ---
--- 3. CÂN BẰNG DỮ LIỆU BẰNG SMOTE CHO TOÀN BỘ TẬP TRAIN ---


/Users/mac/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


53

In [6]:
print("--- 4. HUẤN LUYỆN MÔ HÌNH SVM TOÀN DIỆN ---")
svm_model = LinearSVC(random_state=42, class_weight='balanced', dual=False)
svm_model.fit(X_train_resampled, y_train_resampled)

--- 4. HUẤN LUYỆN MÔ HÌNH SVM TOÀN DIỆN ---


LinearSVC(class_weight='balanced', dual=False, random_state=42)

In [7]:
print("--- 5. ĐÓNG GÓI MÔ HÌNH CHỨA ĐẦY ĐỦ TRI THỨC CỦA 2 THÁNG ---")
# Đóng gói đối tượng chứa tri thức tổng hợp để chuyển giao cho Node.js Backend
joblib.dump(svm_model, 'svm_final_model.joblib')
joblib.dump(scaler, 'data_scaler_combined.joblib')
print("Hoàn tất đóng gói!")

--- 5. ĐÓNG GÓI MÔ HÌNH CHỨA ĐẦY ĐỦ TRI THỨC CỦA 2 THÁNG ---
Hoàn tất đóng gói!
